# 9. MENINGITIS (Missing Values) - Preprocesamiento
**Tipo:** CLASIFICACIÓN MULTICLASE (nivel de riesgo)
**Variable objetivo:** Risk_Level (Low=0, Medium=1, High=2)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# CARGAR DATOS
df = pd.read_csv('mening missing 12.csv')
print(f"Dimensiones: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.head()

In [ ]:
# ============================================================
# ANÁLISIS DE NULOS (Este dataset tiene muchos!)
# ============================================================
print("Nulos por columna:")
print(df.isnull().sum())
print(f"\nTotal nulos: {df.isnull().sum().sum()}")
print(f"Porcentaje: {df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100:.2f}%")

In [ ]:
# ============================================================
# IMPUTACIÓN DE NULOS
# ============================================================
# Numéricas: mediana (más robusta que la media en medicina)
# Categóricas: moda (valor más frecuente)

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['float64', 'int64']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

print(f"Nulos después de imputación: {df.isnull().sum().sum()}")

In [ ]:
# ============================================================
# PREPARAR VARIABLE OBJETIVO (MULTICLASE)
# ============================================================
# Risk_Level: Low, Medium, High → 0, 1, 2
le = LabelEncoder()
df['Risk_Level'] = le.fit_transform(df['Risk_Level'])

y = df['Risk_Level'].values
print(f"Clases: {le.classes_}")
print(f"Distribución: {np.bincount(y)}")

In [ ]:
# ============================================================
# ELIMINAR COLUMNAS NO ÚTILES
# ============================================================
# Patient_ID: no es característica médica
cols_eliminar = ['Patient_ID', 'Risk_Level']
X_df = df.drop(columns=[c for c in cols_eliminar if c in df.columns])

print(f"Columnas para el modelo: {list(X_df.columns)}")

In [ ]:
# ============================================================
# ENCODING DE VARIABLES CATEGÓRICAS
# ============================================================
cat_cols = X_df.select_dtypes(include=['object']).columns.tolist()
print(f"Categóricas: {cat_cols}")

# Binarias (Male/Female, Yes/No) → 0/1
for col in cat_cols:
    unique_vals = X_df[col].nunique()
    if unique_vals == 2:
        X_df[col] = LabelEncoder().fit_transform(X_df[col])

# Multiclase → One-Hot
cat_cols_remaining = X_df.select_dtypes(include=['object']).columns.tolist()
X_encoded = pd.get_dummies(X_df, columns=cat_cols_remaining, drop_first=True)

print(f"\nDimensiones después de encoding: {X_encoded.shape}")

In [ ]:
# ============================================================
# DIVISIÓN DE DATOS
# ============================================================
X = X_encoded.to_numpy()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.333, random_state=42)

print(f"Train: {X_train.shape[0]}")
print(f"Val: {X_val.shape[0]}")
print(f"Test: {X_test.shape[0]}")

In [ ]:
# ============================================================
# NORMALIZACIÓN
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("✅ Datos listos para CLASIFICACIÓN MULTICLASE")
print(f"Características: {X_train.shape[1]}")
print(f"Número de clases: {len(np.unique(y))}")

## MÉTODO TRADICIONAL

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Regresión Logística Multiclase
modelo = LogisticRegression(max_iter=1000, multi_class='multinomial')
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print("=== Regresión Logística Multiclase ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))

## CON PYTORCH (MULTICLASE)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Para multiclase, y debe ser LongTensor
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)  # IMPORTANTE: LongTensor para multiclase
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

num_clases = len(np.unique(y))
print(f"Número de clases: {num_clases}")

In [ ]:
# Red Neuronal MULTICLASE
class ClasificadorRiesgo(nn.Module):
    def __init__(self, input_dim, num_clases):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_clases)  # Salida: num_clases neuronas
            # NO hay Softmax aquí, CrossEntropyLoss lo incluye
        )
    def forward(self, x):
        return self.red(x)

modelo = ClasificadorRiesgo(X_train_t.shape[1], num_clases)

# CrossEntropyLoss para multiclase (incluye Softmax internamente)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)

print(modelo)
print(f"\nInput: {X_train_t.shape[1]} → Output: {num_clases} clases")

In [ ]:
# ============================================================
# EJEMPLO DE ENTRENAMIENTO
# ============================================================
epochs = 50
for epoch in range(epochs):
    modelo.train()
    for X_batch, y_batch in train_loader:
        optimizador.zero_grad()
        outputs = modelo(X_batch)
        loss = criterio(outputs, y_batch)
        loss.backward()
        optimizador.step()
    
    if (epoch + 1) % 10 == 0:
        modelo.eval()
        with torch.no_grad():
            outputs = modelo(X_test_t)
            _, predicciones = torch.max(outputs, 1)
            acc = (predicciones == y_test_t).float().mean()
            print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} - Acc: {acc:.4f}")